# **The Crying Baby**
---

## **Partially Observable MDP**

Lauri et al. state that 
> Noisy sensing, imperfect control, and environment changes are defining characteristics of many real-world robot tasks. The partially observable Markov decision process (POMDP) provides a principled mathematical framework for modeling and solving robot decision and control tasks under uncertainty. A POMDP models a decision-maker, also called the agent, for a system with incomplete state information. At each time step, the agent executes an action that yields some reward depending on the current state and the action, and results in a stochastic transition to a new system state. The agent acquires information about the new state via noisy observations. A solution to a POMDP is a policy that prescribes the agent’s action conditioned on its past history of actions and observations [1].

It is formulated as follows:

$$
M = (S,A,T,R,O,Z,\gamma)
$$

where:

- $S$: State Space
- $A$: Action Space
- $T$: Transition Probabilities
- $R$: Reward Function
- $O$: Observation Space
- $Z$: Observation Model
- $\gamma$: Discount Factor

The agent receives an observation of the current state rather than the true state (potentially imperfect observations). Using past observations the agent builds a **belief** of their underlying state, which can be represented by a probability distribution over the states.

---

## **Crying Baby POMDP**

*The Crying Baby* is a simple decision-making under uncertainty problem. Where the decision-making agent is a caregiver. The baby can be in two states, either **hungry** or **full**. The caregiver however, doesn't know the true current state of the baby. All it sees, are imperfect observations of the environment which are the baby either **crying** or being **quiet**. Based on these observations, it can choose one of the three actions, to **feed** the baby, **ignore** or **sing** to the baby.

The ultimate goal of the caregiver is to keep the baby full with the lowest cost (higher reward). Reflecting on the fact that crying is a strong indicator that the baby is Full or the baby being quiet means the baby is most likely full.

### **State Space**

The state represents the baby's actual condition, which cannot be observed directly.

$$
S = \{\text{Full}, \text{Full}\}
$$

In [7]:
from enum import Enum

In [ ]:
class State(Enum):
    HUNGRY = 0
    FULL = 1

### **Action Space**

The caregiver can choose one of the three actions at each timestep.

$$
A = \{\text{Feed}, \text{Ignore}, \text{Sing}\}
$$

The `Sing` action illustrates an impoertant characteristic of POMDPS, Singing does not change the baby's underlying condition, but it can reveal information about that condition. This additional observation can help the agent form a more accurate estimate of the true state. 

In [10]:
class Action(Enum):
    IGNORE = 0
    SING = 1
    FEED = 2

### **Transition Probabilities**

The transition model describes how the baby's hidden state can change after
taking an action. It is represented by

$$
T(s' \mid s,a) = P(s_{t+1}=s' \mid s_t=s, a_t=a).
$$

Feeding the baby always results in the baby becoming Full.

$$
T(\text{Full} \mid \text{Hungry}, \text{Feed}) = 1
$$

$$
T(\text{Hungry} \mid \text{Hungry}, \text{Feed}) = 0
$$

$$
T(\text{Full} \mid \text{Full}, \text{Feed}) = 1
$$

$$
T(\text{Hungry} \mid \text{Full}, \text{Feed}) = 0
$$

If the baby is hungry and is not fed, it remains hungry.

$$
T(\text{Hungry} \mid \text{Hungry}, a) = 1
$$

$$
T(\text{Full} \mid \text{Hungry}, a) = 0
$$

If the baby is currently not hungry, there is a 10% chance that it becomes hungry during the next time step when the agent chooses either `Sing` or `Ignore`.

$$
T(\text{Hungry} \mid \text{Full}, a) = 0.1
$$

$$
T(\text{Full} \mid \text{Full}, a) = 0.9
$$

where $a \in \{\text{Sing}, \text{Ignore}\}$.

Notice that the transition probabilities are the same for `Ignore` and `Sing`.

$$
T(s' \mid s, Ignore) = T(s' \mid s, Sing)
\\
s, s' \in S
$$

This indicates that ignoring the baby or singing to him has the same impact on the baby's true condition but these actions differ in terms of following observations and rewards which will be discussed later. 


In [13]:
def transition_probability(next_state: State, state: State, action: Action):

    if action == Action.FEED:
        return (1 if next_state == State.FULL else 0)

    if state == State.HUNGRY:
        return (1 if next_state == State.HUNGRY else 0)

    return (0.1 if next_state == State.HUNGRY else 0.9)

In [15]:
# Testing

print("Feeding:")

action = Action.FEED
print(f"T(Full|Hungry, Feed) = {transition_probability(State.FULL, State.HUNGRY, action)}")
print(f"T(Hungry|Hungry, Feed) = {transition_probability(State.HUNGRY, State.HUNGRY, action)}")
print(f"T(Full|Full, Feed) = {transition_probability(State.FULL, State.FULL, action)}")
print(f"T(Hungry|Full, Feed) = {transition_probability(State.HUNGRY, State.FULL, action)}")

print("\na is Singing or Ignoring:")
action = Action.IGNORE
print(f"T(Hungry|Hungry, a) = {transition_probability(State.HUNGRY, State.HUNGRY, action)}")
print(f"T(Full|Hungry, a) = {transition_probability(State.FULL, State.HUNGRY, action)}")
print(f"T(Hungry|Full, a) = {transition_probability(State.HUNGRY, State.FULL, action)}")
print(f"T(Full|Full, a) = {transition_probability(State.FULL, State.FULL, action)}")


Feeding:
T(Full|Hungry, Feed) = 1
T(Hungry|Hungry, Feed) = 0
T(Full|Full, Feed) = 1
T(Hungry|Full, Feed) = 0

a is Singing or Ignoring:
T(Hungry|Hungry, a) = 1
T(Full|Hungry, a) = 0
T(Hungry|Full, a) = 0.1
T(Full|Full, a) = 0.9


### **Reward Function**

The reward model describes the immediate value or cost associated with taking an
action while the system is in a particular state.

The reward function can be written as

$$
R(s,a)
$$

representing the immediate reward received when action $a$ is taken while the
system is in state $s$.

The objective of the agent is not necessarily to maximize the immediate reward
at every step, but to maximize the expected accumulated reward over time:

$$
\mathbb{E}
\left[
\sum_{t=0}^{\infty}
\gamma^t R(s_t,a_t)
\right]
$$

where $\gamma \in [0,1)$ is the discount factor.

For the standard crying baby POMDP , the reward rule is:
- being Hungry: **\(-10\)**
- taking Feed: additional **\(-5\)**
- taking Sing: additional **\(-0.5\)**
- taking Ignore: **no additional action cost**

This reward design encourages the agent to keep the baby full while avoiding unnecessary feeding.

In [17]:
def reward_function(state: State, action: Action):

    reward = 0

    if state == State.HUNGRY:
        reward -= 10

    if action == Action.FEED:
        reward -= 5
    elif action == Action.SING:
        reward -= 0.5

    return reward

---

## **REFERENCES**

**[1]** M. Lauri, D. Hsu, and J. Pajarinen,<br>
        “Partially Observable Markov Decision Processes in Robotics: A Survey,”<br>
        *IEEE Transactions on Robotics*, vol. 39, no. 1, pp. 21–40, Feb. 2023.<br>